# Results NVV-Pipeline Parameter Screening Experiment - Full-GT and Part-GT
- yml: environment.yml
- env: nvv_isolation_pipeline
- last updated: 05.04.2026


## Setup
- Imports
- Config

In [ ]:
import os
import sys
from pathlib import Path

project_root = Path(os.getcwd()).parent
sys.path.append(str(project_root))

import pandas as pd
import matplotlib.pyplot as plt

from config.load_config import load_config
from config.path_factory import print_paths

from evaluation.analysis_loader import load_and_compare_experiments
from evaluation.analysis_tables import (
    build_setting_summary_table,
    build_best_all_screened_values_matrix,
    build_top_region_parameter_values_matrix,
    build_derivative_matrix,
    build_top_k_runs_table
)
from evaluation.analysis_plots import (
    plot_setting_overview,
    plot_four_panel_parameter_summary,
    plot_parameter_pair_heatmaps,
    plot_derivative_group_comparison,
    plot_top_k_runs_per_setting,
)

cfg_path_nvs_full_gt = project_root / "experiments" / "screening" / "config_param_screening_nvs38k_full_gt.yaml"
cfg_path_nvs_part_gt = project_root / "experiments" / "screening" / "config_param_screening_nvs38k_part_gt.yaml"
cfg_path_vocal_part_gt = project_root / "experiments" / "screening" / "config_param_screening_vocal_part_gt.yaml"

experiment_yaml_path = project_root / "experiments" / "screening" / "param_screening.yaml"

config = load_config(cfg_path_nvs_full_gt)
print_paths(cfg_path_nvs_full_gt)

## Collect Screening V1 Experiment Results 
- Experiment specs for comparison
- Combined Views (Tables)

In [ ]:
experiment_specs = [
    {
        "label": "NVS-38K_EN_10_categories | full_gt",
        "cfg_path": cfg_path_nvs_full_gt,
        "dataset_name": "NVS-38K_EN_10_categories",
        "mode": "full_gt",
    },
    {
        "label": "NVS-38K_EN_10_categories | part_gt",
        "cfg_path": cfg_path_nvs_part_gt,
        "dataset_name": "NVS-38K_EN_10_categories",
        "mode": "part_gt",
    },
        {
        "label": "VOCAL_10_categories | part_gt",
        "cfg_path": cfg_path_vocal_part_gt,
        "dataset_name": "VOCAL_10_categories",
        "mode": "part_gt",
    },
]

analysis_bundle = load_and_compare_experiments(
    experiment_specs=experiment_specs,
    experiment_yaml_path=experiment_yaml_path,
    score_fraction=0.99,
    top_n_runs=20,
    param_names=[
        "vad_threshold",
        "vad_min_silence_ms",
        "max_duration",
        "dedup_overlap_ratio",
    ],
    param_pairs=[
        ("vad_threshold", "vad_min_silence_ms"),
    ],
    combo_top_n=20,
    top_k_rq2a_per_run=None,
)

views = analysis_bundle["combined_views"]

In [ ]:
views

In [ ]:
display(views["setting_overview"])


In [ ]:
display(
    views["parameter_value_details"][[
        "setting",
        "dataset_name",
        "mode",
        "parameter",
        "param_value_text",
        "n_runs",
        "mean_primary_score",
        "share_of_top_region",
    ]]
)

In [ ]:
display(views["pair_value_details"])

In [ ]:
views = analysis_bundle["combined_views"]

# --- collect all RQ1 tables from the four loaded experiments ---
# Filter to best_selected_set system rows for per-run comparison
rq1_all = pd.concat(
    [
        bundle["results"]["rq1"]
        for bundle in analysis_bundle["bundles_by_experiment"].values()
    ],
    ignore_index=True,
)
if "system" in rq1_all.columns:
    rq1_all = rq1_all[rq1_all["system"] == "best_selected_set"].copy()

# --- thesis-friendly summary tables ---
setting_summary_tables = build_setting_summary_table(
    views["setting_overview"],
    score_fraction=0.95,
)

best_tables = build_best_all_screened_values_matrix(
    rq1_all,
    dataset_names=[
        "nvs38k_EN_10_categories",
        "vocals_10_categories",
        #"2_files",
    ],
    modes=[
        "full_gt",
        "part_gt",
    ],
)

top_region_parameter_values_matrix = build_top_region_parameter_values_matrix(
    views["parameter_region_summary"]
)

# Keep derivative matrices separated by metric / mode
derivative_matrix_f1 = build_derivative_matrix(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "full_gt"
    ],
    value_col="macro_mean_f1",
)

derivative_matrix_recall = build_derivative_matrix(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "part_gt"
    ],
    value_col="macro_mean_recall",
)

top_k_runs_table = build_top_k_runs_table(
    views["per_setting"],
    top_k=20,
)

print("Setting Summary (Full-GT):")
display(setting_summary_tables["full_gt"])

print("Setting Summary (Part-GT):")
display(setting_summary_tables["part_gt"])

print("\nFull-GT: Best All Screened Values Matrix:")
display(best_tables["full_gt"])

print("\nPart-GT: Best All Screened Values Matrix:")
display(best_tables["part_gt"])

print("\nTop Region Parameter Values Matrix:")
display(top_region_parameter_values_matrix)

print("\nDerivative Matrix (F1, full_gt):")
display(derivative_matrix_f1)

print("\nDerivative Matrix (Recall, part_gt):")
display(derivative_matrix_recall)

print("\nTop K Runs per Setting:")
for setting_name in top_k_runs_table["setting"].unique():
    print(f"\n=== {setting_name} ===")
    display(
        top_k_runs_table[top_k_runs_table["setting"] == setting_name]
        .reset_index(drop=True)
    )

In [ ]:
display(top_k_runs_table)
display(best_tables["full_gt"]) #toDo: kaputt?
display(best_tables["part_gt"]) #toDo: kaputt? Ansicht oben funktioniert

## Experiment Comparison Plots

In [ ]:
fig = plot_setting_overview(
    views["setting_overview"].rename(
        columns={"best_score": "plot_score"}
    ),
    score_col="plot_score",
    title=(
        "Best score per setting\n"
        "(full_gt: macro_mean_f1, part_gt: macro_mean_recall)"
    ),
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="vad_threshold",
    value_col="mean_primary_score",
    title=(
        "VAD threshold across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="vad_min_silence_ms",
    value_col="mean_primary_score",
    title=(
        "VAD minimum silence across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="max_duration",
    value_col="mean_primary_score",
    title=(
        "Maximum NVV duration across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="dedup_overlap_ratio",
    value_col="mean_primary_score",
    title=(
        "Deduplication overlap ratio across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_parameter_pair_heatmaps(
    views["pair_value_details"],
    pair_key="vad_threshold__vad_min_silence_ms",
    value_col="mean_primary_score",
    panel_by="setting",
    title=(
        "VAD threshold × VAD minimum silence across settings\n"
        "Cell value = mean primary score within top region"
    ),
)
plt.show()

fig = plot_derivative_group_comparison(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "full_gt"
    ],
    value_col="macro_mean_f1",
    panel_by="setting",
    title="Audio derivative group comparison (full_gt, metric: macro_mean_f1)",
)
plt.show()

fig = plot_derivative_group_comparison(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "part_gt"
    ],
    value_col="macro_mean_recall",
    panel_by="setting",
    title="Audio derivative group comparison (part_gt, metric: macro_mean_recall)",
)
plt.show()

fig = plot_top_k_runs_per_setting(
    views["per_setting"],
    top_k=10,
    show_param_text=False,
    title=(
        "Top-k runs per setting\n"
        "Bar height = setting-specific primary metric "
        "(full_gt: macro_mean_f1, part_gt: macro_mean_recall)"
    ),
)
plt.show()


In [ ]:
display(top_k_runs_table[top_k_runs_table["setting"] == setting_name]
        .reset_index(drop=True)
    )

fig = plot_setting_overview(
    views["setting_overview"][
        views["setting_overview"]["mode"] == "full_gt"
    ].rename(columns={"best_score": "plot_score"}),
    score_col="plot_score",
    title="Best F1 per setting (full_gt)",
)
plt.show()

fig = plot_setting_overview(
    views["setting_overview"][
        views["setting_overview"]["mode"] == "part_gt"
    ].rename(columns={"best_score": "plot_score"}),
    score_col="plot_score",
    title="Best Recall per setting (part_gt)",
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="vad_threshold",
    value_col="mean_primary_score",
    title=(
        "VAD threshold across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="vad_min_silence_ms",
    value_col="mean_primary_score",
    title=(
        "VAD minimum silence across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="max_duration",
    value_col="mean_primary_score",
    title=(
        "Maximum NVV duration across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="dedup_overlap_ratio",
    value_col="mean_primary_score",
    title=(
        "Deduplication overlap ratio across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_parameter_pair_heatmaps(
    views["pair_value_details"],
    pair_key="vad_threshold__vad_min_silence_ms",
    value_col="mean_primary_score",
    panel_by="setting",
    title=(
        "VAD threshold × VAD minimum silence across settings\n"
        "Cell value = mean primary score within top region"
    ),
)
plt.show()

fig = plot_derivative_group_comparison(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "full_gt"
    ],
    value_col="macro_mean_f1",
    panel_by="setting",
    title="Audio derivative group comparison (full_gt, metric: macro_mean_f1)",
)
plt.show()

fig = plot_derivative_group_comparison(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "part_gt"
    ],
    value_col="macro_mean_recall",
    panel_by="setting",
    title="Audio derivative group comparison (part_gt, metric: macro_mean_recall)",
)
plt.show()

fig = plot_top_k_runs_per_setting(
    views["per_setting"],
    top_k=10,
    show_param_text=False,
    title=(
        "Top-k runs per setting\n"
        "Bar height = setting-specific primary metric "
        "(full_gt: macro_mean_f1, part_gt: macro_mean_recall)"
    ),
)
plt.show()


# Final Parameter Selection for Full Dataset Run (09.04.2026)

**Top-performing region and cutoff definition** 
The top-performing region is defined as all parameter configurations whose primary score lies within 1% of the best observed score. Formally, a configuration is considered part of the top region if its score satisfies:

`score ≥ best_score × 0.99`

The corresponding cutoff score is therefore defined as:

`cutoff_score = best_score × 0.99`

This definition ensures that all configurations with near-optimal performance are considered, avoiding arbitrary rank-based thresholds and allowing for robust identification of parameter sensitivity.

## Runs
3x3x3x3 Parameter Grid = 81 Runs

## Top-Performing Parameter Regions per Setting

| Setting | Top Region Size | vad_threshold | vad_min_silence_ms | max_duration | dedup_overlap_ratio | Primary Metric | Score |
|---|---|---|---|---|---|---|---|
| NVS-38K_EN_10_categories \| full_gt | 9 | 0.20 | 50 | None, 6.5, 13.0 | 0.5, 0.7, 0.9 | macro_mean_f1 | 0.60 |
| NVS-38K_EN_10_categories \| part_gt | 9 | 0.20 | 50 | None, 6.5, 13.0 | 0.5, 0.7, 0.9 | macro_mean_recall | 0.60 |
| VOCAL_10_categories \| part_gt | 27 | 0.20 | 50, 75, 150 | None, 6.5, 13.0 | 0.5, 0.7, 0.9 | macro_mean_recall | 0.229167 |


## Top-Performing Regions with Evaluation Metrics (macro mean)

| Setting | f1 | recall | dice_eos_recall | mean_dice_eos_tp | insertion_rate | Fixed Parameters | Variable Parameters |
|---|---|---|---|---|---|---|---|
| NVS-38K_EN_10_categories \| full_gt | 0.60 | 0.60 | 0.5005 | 0.5005 | 0.40 | vad_threshold=0.20, vad_min_silence_ms=50 | max_duration, dedup_overlap_ratio |
| NVS-38K_EN_10_categories \| part_gt | — | 0.60 | 0.5005 | 0.5005 | 1.20 | vad_threshold=0.20, vad_min_silence_ms=50 | max_duration, dedup_overlap_ratio |
| VOCAL_10_categories \| part_gt | — | 0.229167 | 0.212621 | 0.3728 | ~2.55–2.70 | vad_threshold=0.20 | vad_min_silence_ms, max_duration, dedup_overlap_ratio |

## Selected Parameter

| vad_threshold | vad_min_silence_ms | max_duration | dedup_overlap_ratio |
|---|---|---|---|
| 0.20 | 50 | None | 0.5 |

**vad_threshold = 0.20**  
Across all screening settings, the top-performing region consistently uses `vad_threshold = 0.20`. For `NVS-38K_EN_10_categories`, this value defines the complete top region in both evaluation modes, with the nine best runs achieving **macro_mean_f1 = 0.60** (`full_gt`) and **macro_mean_recall = 0.60** (`part_gt`). Lower-ranked runs with different thresholds show a clear performance decrease. In `VOCAL_10_categories | part_gt`, the complete top region likewise uses `vad_threshold = 0.20`, with a best score of **macro_mean_recall = 0.229167**. Since no alternative threshold appears in any top-performing region, `0.20` is selected as the final value.

**vad_min_silence_ms = 50**  
For `NVS-38K_EN_10_categories`, the entire top-performing region in both evaluation modes uses `vad_min_silence_ms = 50`, with all nine top runs achieving **0.60** in the respective primary metric. Increasing the value to `75` results in a consistent performance drop to **0.50**, and further to **0.40** for `150`. In contrast, for `VOCAL_10_categories | part_gt`, all three tested values (`50`, `75`, `150`) occur equally within the top-performing region and achieve identical recall scores (**0.229167**), indicating no sensitivity to this parameter. Therefore, `50` is selected as the final value, as it is clearly optimal on the NVS-based screening settings while not being disadvantageous on VOCAL.

**max_duration = None**  
The screening does not show any meaningful performance difference between the tested values of `max_duration`. In both NVS-based settings as well as in `VOCAL_10_categories | part_gt`, all three values (`None`, `6.5`, `13.0`) occur in the respective top-performing regions and achieve identical best scores. This indicates that the parameter is not sensitive within the tested range. Therefore, no maximum duration constraint is applied. The choice of `None` is consistent with the recall-oriented design of the pipeline, avoiding the unnecessary exclusion of potentially relevant NVV candidate segments.

**dedup_overlap_ratio = 0.5**  
The screening shows no meaningful sensitivity for `dedup_overlap_ratio`. In all three settings, the values `0.5`, `0.7`, and `0.9` all appear within the top-performing regions and achieve identical best scores. Thus, no value can be selected based on performance differences. The final choice of `0.5` is therefore pragmatic and serves as a consistent default setting rather than reflecting a measurable advantage in the screening results.

Overall, the screening supports a performance-driven selection for the VAD parameters, while `max_duration` and `dedup_overlap_ratio` remain effectively indifferent within the tested ranges and are fixed based on design considerations for the subsequent full evaluation runs.